# Reinforcement Learning — GRPO on a Verifiable-Reward Dataset

Part 3 of the RL series: **Group Relative Policy Optimization (GRPO)** — the algorithm
behind DeepSeek-R1's reasoning training (introduced in DeepSeekMath, 2024). Instead of
training a value network like PPO, GRPO samples a *group* of answers per prompt and
normalizes rewards **within the group** to build the advantage.

**The setup:** a tiny character-level transformer is first taught arithmetic with
supervised fine-tuning (SFT) on a dataset of problems, then improved with GRPO using
**verifiable rewards** — the answer is parsed and checked exactly, no reward model needed.

**Approach:**
1. Build the dataset — arithmetic problems with exactly checkable answers
2. Tiny char-level GPT as the policy
3. SFT warmup → observe that SFT *memorizes* its training split
4. GRPO on held-out prompts where the model is *sometimes* right → amplify success
5. Measure: reward curve, RL-pool accuracy, transfer to unseen problems

*Prerequisites: `ReinforcementLearning.ipynb` (Q-learning) and `DQN_CartPole.ipynb`.*

In [1]:
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

sns.set_style("whitegrid")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch version:", torch.__version__)
print("device:       ", device)

torch version: 2.8.0+cu128
device:        cpu


## 1. The Dataset — Arithmetic Problems with Verifiable Answers

GRPO shines when rewards can be **checked programmatically** (math, code, logic puzzles).
We generate 700 arithmetic questions; the reward function parses the model's answer and
compares it to the true value — no human labels, no reward model.

Answers are zero-padded to 3 digits (`7` → `007`) so every prompt and answer has exactly
the same length — a convenience that keeps the toy implementation simple.

In [2]:
def make_problem(rng):
    op = rng.choice(["+", "-", "*"])
    if op == "*":                        # keep products small (max 12*12 = 144)
        a, b = rng.randint(2, 12), rng.randint(2, 12)
    else:
        a, b = rng.randint(0, 20), rng.randint(0, 20)
        if op == "-" and a < b:
            a, b = b, a                  # keep results non-negative
    answer = {"+": a + b, "-": a - b, "*": a * b}[op]
    return {"prompt": f"Q: {a:02d} {op} {b:02d} =\nA: ", "answer": answer}

rng = random.Random(SEED)
seen, problems = set(), []
while len(problems) < 700:
    p = make_problem(rng)
    key = (p["prompt"], p["answer"])
    if key not in seen:
        seen.add(key)
        problems.append(p)

sft_train = problems[:300]               # supervised warmup only
rl_pool_candidates = problems[300:]      # never seen by SFT — used for GRPO + evaluation
print(f"SFT train: {len(sft_train)}   |   held-out pool: {len(rl_pool_candidates)}")

pd.DataFrame([{"prompt": p["prompt"].replace("\n", "\\n"), "answer": f"{p['answer']:03d}"}
              for p in problems[:8]])

SFT train: 300   |   held-out pool: 400


,prompt,answer
0,Q: 03 * 02 =\nA:,006
1,Q: 06 * 05 =\nA:,030
2,Q: 04 + 03 =\nA:,007
3,Q: 10 * 03 =\nA:,030
4,Q: 08 * 02 =\nA:,016
5,Q: 02 + 06 =\nA:,008
6,Q: 16 + 19 =\nA:,035
7,Q: 17 + 06 =\nA:,023


## 2. Tokenizer & Policy — a Tiny Char-Level GPT

Character-level tokenization keeps the vocabulary tiny (20 tokens); the whole model is
~100K parameters and trains on CPU in minutes.

In [3]:
CHARS = "0123456789+-*=?: \nQA"
stoi = {c: i for i, c in enumerate(CHARS)}
itos = {i: c for c, i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

PROMPT_LEN = len("Q: 00 + 00 =\nA: ")   # 16 — every prompt has this exact length
ANSWER_LEN = 4                           # 3 digits + newline
print(f"Vocab size: {len(CHARS)}, prompt: {PROMPT_LEN} tokens, answer: {ANSWER_LEN} tokens")


class TinyGPT(nn.Module):
    """Minimal GPT: token + learned positional embeddings, causal transformer blocks."""
    def __init__(self, vocab_size, d=128, n_head=4, n_layer=3, max_len=24):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d)
        self.pos = nn.Embedding(max_len, d)
        layer = nn.TransformerEncoderLayer(d, n_head, dim_feedforward=4 * d,
                                           batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, n_layer)
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab_size)

    def forward(self, idx):
        L = idx.size(1)
        mask = torch.triu(torch.full((L, L), float("-inf"), device=idx.device), diagonal=1)
        h = self.tok(idx) + self.pos(torch.arange(L, device=idx.device))
        return self.head(self.ln(self.blocks(h, mask=mask)))


@torch.no_grad()
def generate(model, idx, max_new, temperature=1.0):
    """Batched left-to-right generation (fixed length, lockstep)."""
    model.eval()
    for _ in range(max_new):
        probs = F.softmax(model(idx)[:, -1, :] / temperature, dim=-1)
        idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
    model.train()
    return idx


@torch.no_grad()
def greedy_solve(model, problems):
    """Greedy-decode every prompt; return list of 3-digit answer strings."""
    idx = torch.tensor([encode(p["prompt"]) for p in problems], device=device)
    model.eval()
    for _ in range(ANSWER_LEN):
        idx = torch.cat([idx, model(idx)[:, -1, :].argmax(-1, keepdim=True)], dim=1)
    model.train()
    return [decode(row[PROMPT_LEN:].tolist())[:3] for row in idx]


def accuracy(model, problems):
    outs = greedy_solve(model, problems)
    return np.mean([t.isdigit() and int(t) == p["answer"] for t, p in zip(outs, problems)])


def per_token_logp(model, seq):
    """Log-prob of every token given its context. Returns (logp[N, L-1], answer mask)."""
    logits = model(seq[:, :-1])
    logp = F.log_softmax(logits, dim=-1)
    tok_logp = logp.gather(-1, seq[:, 1:].unsqueeze(-1)).squeeze(-1)
    mask = torch.zeros_like(tok_logp)
    mask[:, PROMPT_LEN - 1:] = 1.0         # only the generated answer tokens count
    return tok_logp, mask


policy = TinyGPT(len(CHARS)).to(device)
print(f"Policy parameters: {sum(p.numel() for p in policy.parameters()):,}")

Vocab size: 20, prompt: 16 tokens, answer: 4 tokens
Policy parameters: 603,284


/usr/local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


## 3. Stage 1 — SFT Warmup (and a First Look at Memorization)

RL can't bootstrap from random noise: the model must at least produce the answer *format*
(`042\n`) before rewards become informative. A supervised pass on the SFT split teaches
format and rough arithmetic.

Watch the gap: after 800 steps the model is nearly perfect **on its SFT split** but much
weaker on unseen problems — SFT mostly *memorizes*. This is the gap GRPO will work on.

In [4]:
opt = torch.optim.Adam(policy.parameters(), lr=3e-3)
SFT_STEPS = 800
for step in range(1, SFT_STEPS + 1):
    batch = random.sample(sft_train, 64)
    seq = torch.tensor([encode(p["prompt"] + f"{p['answer']:03d}\n") for p in batch],
                       device=device)
    tok_logp, mask = per_token_logp(policy, seq)
    loss = -(tok_logp * mask).sum() / mask.sum()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 200 == 0:
        print(f"SFT step {step:>3}/{SFT_STEPS} — loss {loss.item():.3f}, "
              f"train acc {accuracy(policy, sft_train[:100]):.0%}, "
              f"held-out acc {accuracy(policy, rl_pool_candidates[:100]):.0%}")

SFT step 200/800 — loss 0.461, train acc 42%, held-out acc 10%


SFT step 400/800 — loss 0.268, train acc 82%, held-out acc 7%


SFT step 600/800 — loss 0.231, train acc 92%, held-out acc 14%


SFT step 800/800 — loss 0.091, train acc 96%, held-out acc 13%


## 4. Choosing the GRPO Prompt Set

A group only yields signal if its rewards **vary** — if the model always (or never)
answers a prompt correctly, all advantages are ~0 and nothing is learned. Real GRPO
pipelines therefore filter prompts to the model's *frontier*. We do the same: sample
16 answers per held-out prompt and keep prompts with an intermediate success rate.

In [5]:
def reward_of(completion, true_answer):
    """Verifiable reward: 1.0 exact match, 0.1 well-formed but wrong, 0.0 garbage."""
    text = completion[:3]
    if not text.isdigit():
        return 0.0
    return 1.0 if int(text) == true_answer else 0.1


def sampled_success(model, problem, G=16):
    idx = torch.tensor([encode(problem["prompt"])] * G, device=device)
    outs = generate(model, idx, ANSWER_LEN, temperature=1.0)
    return np.mean([reward_of(decode(o[PROMPT_LEN:].tolist()), problem["answer"])
                    for o in outs])


rates = [(sampled_success(policy, p), p) for p in rl_pool_candidates]
intermediate = [p for r, p in rates if 0.2 <= r <= 0.85]

rl_prompts = intermediate[:48]           # GRPO trains on these
transfer_set = intermediate[48:148]      # held-out: same difficulty, never trained on
print(f"Held-out problems: {len(rates)}  |  intermediate: {len(intermediate)}")
print(f"GRPO prompt set: {len(rl_prompts)}  |  transfer set: {len(transfer_set)}")
print(f"Mean sampled success on GRPO prompts: "
      f"{np.mean([r for r, p in rates if 0.2 <= r <= 0.85][:48]):.2f}")

Held-out problems: 400  |  intermediate: 75
GRPO prompt set: 48  |  transfer set: 27
Mean sampled success on GRPO prompts: 0.45


## 5. The GRPO Algorithm

For each prompt $x$, sample a **group** of $G$ answers $y_1, \dots, y_G$ from the current
policy, score each with the verifiable reward, and normalize *within the group*:

$$A_i = \frac{r_i - \operatorname{mean}(r_1, \dots, r_G)}{\operatorname{std}(r_1, \dots, r_G)}$$

Then update with a PPO-style clipped surrogate plus a KL penalty toward a frozen
**reference model** (the SFT checkpoint):

$$\mathcal{L} = -\mathbb{E}\Big[\min\big(\rho_i A_i,\; \operatorname{clip}(\rho_i, 1-\epsilon, 1+\epsilon) A_i\big)\Big]
+ \beta\, \mathrm{KL}(\pi_\theta \,\|\, \pi_{\text{ref}}),
\qquad \rho_i = \frac{\pi_\theta(y_i \mid x)}{\pi_{\text{old}}(y_i \mid x)}$$

**Why no critic?** PPO learns a value network to estimate the baseline; GRPO replaces it
with the group mean — one less network, and a perfect fit for verifiable tasks where
sampling many answers per prompt is cheap.

In [6]:
def grpo_step(policy, ref_model, opt, prompts, answers,
              G=8, clip_eps=0.2, beta=0.04, temperature=1.0):
    B = len(prompts)
    # --- 1. Sample a group of G answers per prompt from the current policy ---
    idx = torch.tensor([encode(p) for p in prompts for _ in range(G)], device=device)
    seq = generate(policy, idx, ANSWER_LEN, temperature=temperature)

    # --- 2. Verifiable rewards + group-relative advantages ---
    rewards = torch.tensor(
        [reward_of(decode(row[PROMPT_LEN:].tolist()), answers[i // G])
         for i, row in enumerate(seq)], device=device)
    R = rewards.view(B, G)
    A = ((R - R.mean(dim=1, keepdim=True))
         / (R.std(dim=1, keepdim=True) + 1e-4)).flatten()

    # --- 3. Frozen log-probs of the sampling policy (old) and the reference ---
    with torch.no_grad():
        logp_old, mask = per_token_logp(policy, seq)
        logp_ref, _ = per_token_logp(ref_model, seq)

    # --- 4. Clipped surrogate + KL-to-reference penalty ---
    logp, mask = per_token_logp(policy, seq)
    ratio = torch.exp(logp - logp_old)
    adv = A.unsqueeze(1)
    surr1 = ratio * adv
    surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv
    pg_loss = -(torch.min(surr1, surr2) * mask).sum() / mask.sum()
    # k1 KL estimator, differentiated correctly through logp
    kl_est = ((logp - logp_ref) * mask).sum(dim=1) / mask.sum(dim=1)
    kl_term = (kl_est.detach().unsqueeze(1) * logp * mask).sum() / mask.sum()

    loss = pg_loss + beta * kl_term
    opt.zero_grad(); loss.backward(); opt.step()
    return rewards.mean().item()

## 6. Stage 2 — GRPO Training

Every iteration uses **all 48 RL prompts** (small set — RL prompt pools are typically far
smaller than the SFT corpus). The KL anchor (β = 0.04) keeps the policy from drifting off
the SFT format while it climbs.

In [7]:
ref_model = copy.deepcopy(policy)        # frozen SFT checkpoint
for p in ref_model.parameters():
    p.requires_grad_(False)

grpo_policy = copy.deepcopy(policy)      # the model GRPO will train
opt = torch.optim.Adam(grpo_policy.parameters(), lr=2e-4)

GRPO_ITERS = 75
reward_history, sampled_history = [], []

pool_acc_before = accuracy(grpo_policy, rl_prompts)
transfer_before = accuracy(grpo_policy, transfer_set)

for it in range(1, GRPO_ITERS + 1):
    r = grpo_step(grpo_policy, ref_model, opt,
                  [p["prompt"] for p in rl_prompts], [p["answer"] for p in rl_prompts])
    reward_history.append(r)
    if it % 15 == 0:
        exact = np.mean([sampled_success(grpo_policy, p) == 1.0 for p in rl_prompts])
        sampled_history.append((it, exact))
        print(f"iter {it:>3}/{GRPO_ITERS} — mean reward {np.mean(reward_history[-15:]):.3f}, "
              f"sampled exact {exact:.0%}")

pool_acc_after = accuracy(grpo_policy, rl_prompts)
transfer_after = accuracy(grpo_policy, transfer_set)

iter  15/75 — mean reward 0.480, sampled exact 4%


iter  30/75 — mean reward 0.537, sampled exact 17%


iter  45/75 — mean reward 0.571, sampled exact 12%


iter  60/75 — mean reward 0.580, sampled exact 19%


iter  75/75 — mean reward 0.590, sampled exact 27%


## 7. Results

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

window = 15
axes[0].plot(np.convolve(reward_history, np.ones(window) / window, mode="valid"))
axes[0].set_title(f"GRPO Mean Reward (moving average, window={window})")
axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("Mean reward")

if sampled_history:
    xs, ys = zip(*sampled_history)
    axes[1].plot(xs, [y * 100 for y in ys], marker="o")
axes[1].set_title("Sampled Exact-Match Rate on RL Prompts")
axes[1].set_xlabel("Iteration"); axes[1].set_ylabel("Exact match (%)")

plt.tight_layout()
plt.show()

print(f"RL prompts (greedy accuracy)  : {pool_acc_before:.0%}  ->  {pool_acc_after:.0%}")
print(f"Transfer set (never trained)  : {transfer_before:.0%}  ->  {transfer_after:.0%}")

<Figure size 1300x400 with 2 Axes>

RL prompts (greedy accuracy)  : 40%  ->  60%
Transfer set (never trained)  : 56%  ->  56%


In [9]:
print("Sample generations — SFT model vs. after GRPO:")
sft_outs = greedy_solve(ref_model, rl_prompts[:8])
grpo_outs = greedy_solve(grpo_policy, rl_prompts[:8])
for p, before, after in zip(rl_prompts[:8], sft_outs, grpo_outs):
    true = f"{p['answer']:03d}"
    mark = lambda o: "✓" if o == true else "✗"
    print(f"  {p['prompt'].splitlines()[0][3:]:>10}  true: {true}   "
          f"SFT: {before} {mark(before)}   GRPO: {after} {mark(after)}")

Sample generations — SFT model vs. after GRPO:
   12 * 02 =  true: 024   SFT: 024 ✓   GRPO: 024 ✓
   04 + 17 =  true: 021   SFT: 019 ✗   GRPO: 021 ✓
   19 + 17 =  true: 036   SFT: 036 ✓   GRPO: 036 ✓
   04 - 03 =  true: 001   SFT: 001 ✓   GRPO: 001 ✓
   06 * 09 =  true: 054   SFT: 054 ✓   GRPO: 060 ✗
   19 - 16 =  true: 003   SFT: 003 ✓   GRPO: 003 ✓
   19 + 08 =  true: 027   SFT: 023 ✗   GRPO: 023 ✗
   02 + 16 =  true: 018   SFT: 013 ✗   GRPO: 028 ✗


## 8. Conclusion

- **GRPO** fine-tunes a generative policy with RL *without a value network*: the
  baseline for each sample is simply the mean reward of its group.
- **Verifiable rewards** remove the reward model: the loop is sample a group → check
  answers → normalize advantages → clipped update with a KL anchor.
- **Prompt selection matters**: groups need reward variance, so we train where the model
  is *sometimes* right — real pipelines filter prompts the same way.
- GRPO **amplifies behaviors the model can already sample**; it cannot create capability
  from zero — prompts where all samples fail give no gradient.
- The **KL penalty** keeps gains from collapsing into prompt-set overfitting; here it
  lets improvement transfer to unseen problems of similar difficulty.

**Real-world counterpart:** at scale this runs on LLMs with datasets like GSM8K or
competition math, using tools such as Hugging Face TRL's `GRPOTrainer`.

*Series recap: tabular Q-learning → DQN (value-based RL with function approximation) →
GRPO (policy-gradient RL for generative models).*